# P00b — EM-DAT Disaster Data

Load and explore the EM-DAT international disaster database. Convert to Arrow for downstream use.

**Source:** [EM-DAT](https://www.emdat.be/) — 27,536 events, 1900–2026, 231 countries (includes historical pre-2000 events with known reporting bias)

In [1]:
include("phase00b/functions/load_phase00b.jl")

## 1. Load Raw Events

In [16]:
events = emdat_load_events();

In [3]:
describe(events)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,Type
1,dis_no,,1950-0001-PAK,,2026-0179-MWI,0,String
2,disaster_group,,Natural,,Technological,0,String
3,disaster_type,,Air,,Wildfire,0,String
4,disaster_subtype,,Air,,Worms infestation,0,String
5,iso3,,AFG,,ZWE,0,String
6,country,,Afghanistan,,Zimbabwe,0,String
7,subregion,,Australia and New Zealand,,Western Europe,0,String
8,region,,Africa,,Oceania,0,String
9,start_year,2002.22,1950.0,2004.0,2026.0,0,Float64


In [4]:
propertynames(events)

16-element Vector{Symbol}:
 :dis_no
 :disaster_group
 :disaster_type
 :disaster_subtype
 :iso3
 :country
 :subregion
 :region
 :start_year
 :start_month
 :end_year
 :total_deaths
 :total_affected
 :latitude
 :longitude
 :is_historic

In [5]:
first(events, 5)

Row,dis_no,disaster_group,disaster_type,disaster_subtype,iso3,country,subregion,region,start_year,start_month,end_year,total_deaths,total_affected,latitude,longitude,is_historic
,String,String,String,String,String,String,String,String,Float64,Float64?,Float64,Float64?,Float64?,Float64?,Float64?,Bool
1,1950-0001-PAK,Natural,Flood,Riverine flood,PAK,Pakistan,Southern Asia,Asia,1950.0,missing,1950.0,2900.0,missing,missing,missing,true
2,1950-0005-PER,Natural,Earthquake,Ground movement,PER,Peru,Latin America and the Caribbean,Americas,1950.0,5.0,1950.0,83.0,200.0,-13.5,-72.0,true
3,1950-0006-IND,Natural,Flood,Riverine flood,IND,India,Southern Asia,Asia,1950.0,7.0,1950.0,45.0,25000.0,missing,missing,true
4,1950-0007-CHN,Natural,Flood,Flood (General),CHN,China,Eastern Asia,Asia,1950.0,8.0,1950.0,500.0,1.0e7,missing,missing,true
5,1950-0008-IND,Natural,Earthquake,Ground movement,IND,India,Southern Asia,Asia,1950.0,8.0,1950.0,1500.0,missing,28.5,96.5,true


## 2. Coverage

In [6]:
# Year range, country count, event counts by disaster group
println("Events:    $(nrow(events))")
println("Years:     $(minimum(skipmissing(events.start_year))) – $(maximum(skipmissing(events.start_year)))")
println("Countries: $(length(unique(events.iso3)))")
println()
combine(groupby(events, :disaster_group), nrow => :n_events) |> df -> sort(df, :n_events, rev=true)

Events:    26660
Years:     1950.0 – 2026.0
Countries: 231



Row,disaster_group,n_events
,String,Int64
1,Natural,17169
2,Technological,9491


In [7]:
# Top disaster types
combine(groupby(events, :disaster_type), nrow => :n_events) |> df -> sort(df, :n_events, rev=true) |> df -> first(df, 15)

Row,disaster_type,n_events
,String,Int64
1,Flood,6126
2,Storm,4881
3,Road,3011
4,Water,1688
5,Epidemic,1500
6,Earthquake,1397
7,Air,954
8,Mass movement (wet),856
9,Fire (Miscellaneous),814


In [8]:
# Events per year — check for pre-2000 reporting bias
valid_years = dropmissing(events, :start_year)
year_counts = combine(groupby(valid_years, :start_year), nrow => :n_events)
sort!(year_counts, :start_year)
println("Events by decade:")
for decade in [1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
    n = sum(year_counts[coalesce.(year_counts.start_year .>= decade, false) .& 
                        coalesce.(year_counts.start_year .< decade + 10, false), :n_events])
    println("  $(decade)s: $n")
end

Events by decade:
  1950s: 390
  1960s: 694
  1970s: 1136
  1980s: 2690
  1990s: 4981
  2000s: 7540
  2010s: 5705
  2020s: 3524


In [9]:
# Historic flag distribution
combine(groupby(events, :is_historic), nrow => :n_events)

Row,is_historic,n_events
,Bool,Int64
1,false,16769
2,true,9891


## 2b. Country Code Alignment with QoG

In [10]:
using Arrow

# Load augmented QoG country codes (Phase 0 output)
qog = DataFrame(Arrow.Table("data/qog_std_ts_jan25_aug.arrow"))
qog_codes = Set(unique(skipmissing(qog.ident_ccodealp)))
emdat_codes = Set(unique(events.iso3))

# Find mismatches
in_emdat_not_qog = sort(collect(setdiff(emdat_codes, qog_codes)))
in_qog_not_emdat = sort(collect(setdiff(qog_codes, emdat_codes)))
matched = length(intersect(emdat_codes, qog_codes))

println("QoG countries:    $(length(qog_codes))")
println("EM-DAT countries: $(length(emdat_codes))")
println("Matched:          $matched")
println()
println("In EM-DAT but NOT in QoG ($(length(in_emdat_not_qog))):")
for c in in_emdat_not_qog
    name = first(filter(r -> r.iso3 == c, events)).country
    println("  $c — $name")
end
println()
println("In QoG but NOT in EM-DAT ($(length(in_qog_not_emdat))):")
for c in in_qog_not_emdat
    println("  $c")
end

QoG countries:    202
EM-DAT countries: 231
Matched:          196

In EM-DAT but NOT in QoG (35):
  AIA — Anguilla
  ANT — Netherlands Antilles
  ASM — American Samoa
  AZO — Azores Islands
  BLM — Saint Barthélemy
  BMU — Bermuda
  COK — Cook Islands
  CUW — Curaçao
  CYM — Cayman Islands
  DFR — Germany Federal Republic
  GLP — Guadeloupe
  GUF — French Guiana
  GUM — Guam
  HKG — China, Hong Kong Special Administrative Region
  MAC — China, Macao Special Administrative Region
  MAF — Saint Martin (French Part)
  MNP — Northern Mariana Islands
  MSR — Montserrat
  MTQ — Martinique
  MYT — Mayotte
  NCL — New Caledonia
  NIU — Niue
  PRI — Puerto Rico
  PSE — State of Palestine
  PYF — French Polynesia
  REU — Réunion
  SHN — Saint Helena
  SPI — Canary Islands
  SXM — Sint Maarten (Dutch part)
  TCA — Turks and Caicos Islands
  TKL — Tokelau
  VGB — British Virgin Islands
  VIR — United States Virgin Islands
  WLF — Wallis and Futuna Islands
  YMN — Yemen Arab Republic

In QoG but NO

## 3. Impact Measures — Missingness

In [ ]:
# Impact column completeness (after 40% threshold drops — see phase0b docs)
# Kept: total_deaths (~80%), total_affected (~67%)
# Dropped: no_injured, no_affected, no_homeless, total_damage_adj_k, magnitude
impact_cols = [:total_deaths, :total_affected]
for col in impact_cols
    n_present = count(!ismissing, events[!, col])
    pct = round(100 * n_present / nrow(events), digits=1)
    println("  $(rpad(string(col), 25)) $(n_present) / $(nrow(events))  ($pct%)")
end

In [ ]:
# Temporal missingness check (retained columns only)
pre = filter(r -> coalesce(r.start_year < 2000, false), events)
post = filter(r -> coalesce(r.start_year >= 2000, false), events)

println("Column                    Pre-2000 (n=$(nrow(pre)))    Post-2000 (n=$(nrow(post)))")
println("─"^75)
for col in [:total_deaths, :total_affected]
    pre_pct = round(100 * count(!ismissing, pre[!, col]) / nrow(pre), digits=1)
    post_pct = round(100 * count(!ismissing, post[!, col]) / nrow(post), digits=1)
    println("  $(rpad(string(col), 25)) $(lpad(string(pre_pct), 5))%              $(lpad(string(post_pct), 5))%")
end

## 4. Aggregate to Country-Year

In [13]:
cy = emdat_aggregate_country_year(events)
describe(cy)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,Type
1,iso3,,AFG,,ZWE,0,String
2,country,,Afghanistan,,Zimbabwe,0,String
3,year,1998.21,1950,2001.0,2026,0,Int64
4,n_events,3.85316,1,2.0,101,0,Int64
5,n_natural,2.48143,0,1.0,43,0,Int64
6,n_technological,1.37173,0,0.0,71,0,Int64
7,sum_deaths,1435.42,1.0,61.0,2.00197e6,1159,"Union{Missing, Float64}"
8,sum_affected,1.56172e6,1.0,15187.5,3.46562e8,1199,"Union{Missing, Float64}"


In [14]:
# Top 10 country-years by event count
sort(cy, :n_events, rev=true) |> df -> first(df, 10)

Row,iso3,country,year,n_events,n_natural,n_technological,sum_deaths,sum_affected
,String,String,Int64,Int64,Int64,Int64,Float64?,Float64?
1,CHN,China,2005,101,31,70,3108.0,8.39559e7
2,CHN,China,2004,95,24,71,2545.0,5.31022e7
3,CHN,China,2001,85,33,52,2315.0,4.03716e7
4,CHN,China,2002,85,28,57,2847.0,2.85291e8
5,CHN,China,2003,82,27,55,2752.0,2.19644e8
6,CHN,China,2006,81,37,44,3081.0,8.8747e7
7,CHN,China,2000,76,28,48,2579.0,2.53936e7
8,CHN,China,2008,64,30,34,89233.0,1.36909e8
9,IND,India,2005,64,31,33,6324.0,2.86705e7


## 5. Write Arrow

Uses `emdat_process_all()` from the Julia function file — loads, aggregates, and writes both Arrow files.

In [ ]:
result = emdat_process_all()
println("\n  Events: $(nrow(result.events)) rows")
println("  Country-year: $(nrow(result.country_year)) rows")